# Replicated credible sets

A GWAS credible set is replicated if its lead variant is associated with the same disease
in at least two independent study records; a molQTL credible set if its lead variant is
associated with the same gene at least twice. Methods "Replication of CSs".

Writes `replicated_gwas_cs`, `replicated_molqtl_cs`.

In [ ]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

In [ ]:
cs = StudyLocus.from_parquet(session, paper.release("credible_set")).df.select(
    "studyId", "variantId", "studyType", "studyLocusId"
)
si = StudyIndex.from_parquet(session, paper.release("study")).df.select(
    "studyId", "diseaseIds", "geneId", "biosampleId", "pubmedId", "cohorts", "ldPopulationStructure"
)
loci = cs.join(si, on="studyId", how="left").cache()
print("credible sets:", loci.count())

## GWAS — same lead variant and disease seen at least twice

In [ ]:
gwas = loci.filter(f.col("studyType") == "gwas").withColumn("diseaseId", f.explode("diseaseIds")).cache()

replicated_pairs = (
    gwas.select("variantId", "diseaseId", "cohorts", "pubmedId", "ldPopulationStructure")
    .dropDuplicates()
    .groupBy("variantId", "diseaseId")
    .agg(f.count("*").alias("count"))
    .filter(f.col("count") >= 2)
)
replicated_gwas = (
    gwas.join(replicated_pairs, on=["variantId", "diseaseId"], how="inner").select("studyLocusId").distinct()
)
replicated_gwas.write.mode("overwrite").parquet(paper.derived("replicated_gwas_cs"))
print("replicated GWAS CSs:", session.spark.read.parquet(paper.derived("replicated_gwas_cs")).count())

## molQTL — same lead variant and gene seen at least twice

In [ ]:
molqtl = loci.filter(f.col("studyType") != "gwas").cache()

replicated_pairs = molqtl.groupBy("variantId", "geneId").agg(f.count("*").alias("count")).filter(f.col("count") >= 2)
replicated_molqtl = (
    molqtl.join(replicated_pairs, on=["variantId", "geneId"], how="inner").select("studyLocusId").distinct()
)
replicated_molqtl.write.mode("overwrite").parquet(paper.derived("replicated_molqtl_cs"))
print("replicated molQTL CSs:", session.spark.read.parquet(paper.derived("replicated_molqtl_cs")).count())

In [ ]:
for name, old in [
    ("replicated_gwas_cs", "list_of_gwas_replicated_CSs.parquet"),
    ("replicated_molqtl_cs", "list_of_molqtls_replicated_CSs.parquet"),
]:
    new = session.spark.read.parquet(paper.derived(name))
    ref = session.spark.read.parquet(paper.baseline(old))
    print(
        name,
        "new:",
        new.count(),
        "baseline:",
        ref.count(),
        "symmetric difference:",
        new.union(ref).distinct().count() - new.intersect(ref).count(),
    )